In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
#------------------------------------------
# 1. Load and Prepare the Data
#------------------------------------------


TrainingDF = pd.read_csv('./training_event_summary_with_m_inv.csv', sep =',', index_col=False)
X_NonExotic = pd.read_csv('./real_data_without_pentaquark.csv', sep =',', index_col=False)
X_Exotic = pd.read_csv('./real_data_with_pentaquark.csv', sep =',', index_col=False)


X=TrainingDF[['proton_PID_sum','kaon_PID_sum','pion_PID_sum','electron_PID_sum','muon_PID_sum', 'n_tracks']].values
# Extract mass separately
mass = TrainingDF['m_inv_pair'].values

# Create a train/test split

X_train, X_test, mass_train, mass_test = train_test_split(X, mass, test_size=0.05, random_state=42)

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(X_train.shape, X_test.shape)

import tensorflow as tf
print("TensorFlow version:", tf.version)
print("Available devices:", tf.config.list_physical_devices())
#tf.debugging.set_log_device_placement(True)

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" # Disable GPU (if any)
#------------------------------------------
# 2. Define the Autoencoder Model
#------------------------------------------

input_dim = X_train.shape[1] # should be 6
encoding_dim = 4 # latent space dimension
# Input layer

input_layer = Input(shape=(input_dim,))
# Encoding layer (compression)

encoded = Dense(encoding_dim, activation='relu')(input_layer)
# Decoding layer (reconstruction)

decoded = Dense(input_dim, activation='sigmoid')(encoded)
# Autoencoder model

autoencoder = Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(optimizer='adam', loss='mse')

from tensorflow.keras.optimizers import Adam
autoencoder.compile(optimizer=Adam(learning_rate=0.01), loss='mse')
#autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse') # best, 185 epochs automatic stop
#------------------------------------------
#3. Training the Autoencoder
#------------------------------------------

early_stopping = EarlyStopping(
monitor='val_loss',
patience=2,
restore_best_weights=True
)

history = autoencoder.fit(X_train, X_train,
epochs=500,
# epochs=500,
batch_size=256,
# batch_size=64,
shuffle=True,
validation_data=(X_test, X_test),
verbose=2,
callbacks=[early_stopping]
)
#with tf.device('/CPU:0'):
#    history = autoencoder.fit(X_train, X_train,
#    epochs=500,
#    batch_size=64,
#    validation_data=(X_test, X_test),
#    callbacks=[early_stopping],
#    verbose=2)
#print(f"Training stopped at epoch {len(history.history['loss'])}")

print(f"Training stopped at epoch {len(history.history['loss'])}")

I0000 00:00:1780923941.917675   17976 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780923941.918068   17976 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780923941.951715   17976 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780923942.711076   17976 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

(114000, 6) (6000, 6)
TensorFlow version: <module 'tensorflow._api.v2.version' from '/home/student/KPacut/.venv/lib/python3.12/site-packages/tensorflow/_api/v2/version/__init__.py'>
Available devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
Epoch 1/500


E0000 00:00:1780923943.157662   17976 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


446/446 - 1s - 2ms/step - loss: 0.0364 - val_loss: 0.0053
Epoch 2/500
446/446 - 0s - 635us/step - loss: 0.0047 - val_loss: 0.0035
Epoch 3/500
446/446 - 0s - 780us/step - loss: 0.0024 - val_loss: 0.0018
Epoch 4/500
446/446 - 0s - 705us/step - loss: 0.0016 - val_loss: 0.0014
Epoch 5/500
446/446 - 0s - 744us/step - loss: 0.0014 - val_loss: 0.0013
Epoch 6/500
446/446 - 0s - 624us/step - loss: 0.0013 - val_loss: 0.0013
Epoch 7/500
446/446 - 0s - 644us/step - loss: 0.0013 - val_loss: 0.0012
Epoch 8/500
446/446 - 0s - 856us/step - loss: 0.0013 - val_loss: 0.0012
Epoch 9/500
446/446 - 0s - 731us/step - loss: 0.0012 - val_loss: 0.0011
Epoch 10/500
446/446 - 0s - 653us/step - loss: 0.0012 - val_loss: 0.0011
Epoch 11/500
446/446 - 0s - 805us/step - loss: 0.0011 - val_loss: 0.0011
Epoch 12/500
446/446 - 0s - 625us/step - loss: 0.0011 - val_loss: 0.0010
Epoch 13/500
446/446 - 0s - 621us/step - loss: 0.0011 - val_loss: 0.0010
Epoch 14/500
446/446 - 0s - 610us/step - loss: 0.0011 - val_loss: 0.0011
E